# AAAI Supplemental Code Notebook

This notebook combines training, generation, and evaluation steps used in the paper.

In [ ]:
!pip install transformers==4.41.2 datasets==2.20.0 pandas==2.2.2 torch==2.2.2 bert-score==0.3.13 nltk==3.8.1

## Baseline Model Training (mBART)

In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import MBart50TokenizerFast, MBartForConditionalGeneration, Trainer, TrainingArguments

df = pd.read_csv('/content/drive/MyDrive/Benchmarking/Multitarget-CONAN.csv')
dataset = Dataset.from_pandas(df)

model_name = 'facebook/mbart-large-50'
tokenizer = MBart50TokenizerFast.from_pretrained(model_name)
model = MBartForConditionalGeneration.from_pretrained(model_name)
tokenizer.src_lang = 'en_XX'
tokenizer.tgt_lang = 'en_XX'

def tokenize_function(example):
    input_text = 'generate persian counter speech : ' + example['HATE_SPEECH']
    target_text = example['COUNTER_NARRATIVE']
    model_inputs = tokenizer(input_text, max_length=128, padding='max_length', truncation=True)
    labels = tokenizer(text_target=target_text, max_length=128, padding='max_length', truncation=True)
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

tokenized_dataset = dataset.map(tokenize_function, batched=False, remove_columns=dataset.column_names)

args = TrainingArguments(
    output_dir='./mbart-counter-speech',
    per_device_train_batch_size=8,
    num_train_epochs=5,
    logging_dir='./logs',
    save_total_limit=1,
    eval_strategy='no',
    report_to=[]
)

trainer = Trainer(model=model, args=args, train_dataset=tokenized_dataset)
trainer.train()

## PersianMind Counterspeech Generation

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch, re

persianmind_model_name = 'universitytehran/PersianMind-v1.0'
tokenizer = AutoTokenizer.from_pretrained(persianmind_model_name)
model = AutoModelForCausalLM.from_pretrained(persianmind_model_name)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)

input_data = pd.read_csv('/content/drive/MyDrive/Benchmarking/generated_persian_counterspeech.csv')

def clean_response(text):
    text = re.sub(r'[^\u0600-\u06FF\s]', '', text)
    return ' '.join(text.split()[:30])

def generate_counterspeech(hate_speech):
    prompt = f'به این گفتار نفرت‌آمیز با پاسخی مثبت، محترمانه و کوتاه (حداکثر 30 کلمه) پاسخ دهید: {hate_speech}'
    inputs = tokenizer(prompt, return_tensors='pt', padding=True, truncation=True, max_length=512)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    outputs = model.generate(**inputs, max_new_tokens=50, temperature=0.7, top_p=0.9, do_sample=True)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return clean_response(response[len(prompt):].strip())

responses = []
for _, row in input_data.iterrows():
    responses.append({'hate_speech': row['Hate_Speech'], 'counterspeech': generate_counterspeech(row['Hate_Speech'])})

pd.DataFrame(responses).to_csv('persian_counterspeech_output.csv', index=False, encoding='utf-8')

## Automatic Evaluation (Perplexity, BERTScore, Distinct-n)

In [ ]:
import math
from transformers import AutoTokenizer, AutoModelWithLMHead
from bert_score import score as bert_score
from nltk import ngrams

df = pd.read_csv('/content/drive/MyDrive/ParsCN.csv')
n_samples = 125
df_sampled = df.groupby('Annotator').apply(lambda x: x.sample(n=min(n_samples, len(x)), random_state=42)).reset_index(drop=True)

per_tokenizer = AutoTokenizer.from_pretrained('HooshvareLab/gpt2-fa')
per_model = AutoModelWithLMHead.from_pretrained('HooshvareLab/gpt2-fa')

def calculate_perplexity(text):
    inputs = per_tokenizer(text, return_tensors='pt', truncation=True)
    with torch.no_grad():
        outputs = per_model(**inputs, labels=inputs['input_ids'])
    return math.exp(outputs.loss.item())

df_sampled['Perplexity'] = df_sampled['Counter_Narrative'].apply(calculate_perplexity)

P, R, F1 = bert_score(df_sampled['Counter_Narrative'].tolist(), df_sampled['Hate_Speech'].tolist(),
                      model_type='bert-base-multilingual-cased', lang='fa')
df_sampled['BERTScore_P'], df_sampled['BERTScore_R'], df_sampled['BERTScore_F1'] = P, R, F1

def distinct_ngrams(texts, n):
    ngram_list = [gram for text in texts for gram in ngrams(text.split(), n)]
    return len(set(ngram_list)) / len(ngram_list) if ngram_list else 0

distinct_results = []
for annotator in df_sampled['Annotator'].unique():
    subset = df_sampled[df_sampled['Annotator'] == annotator]['Counter_Narrative'].tolist()
    distinct_results.append((annotator, distinct_ngrams(subset, 1), distinct_ngrams(subset, 2)))

pd.DataFrame(distinct_results, columns=['Annotator', 'Distinct-1', 'Distinct-2']).to_csv('Distinct_scores_125each.csv', index=False)
df_sampled.to_csv('ParsCN_125each_with_metrics.csv', index=False)

## Toxicity Evaluation

In [ ]:
from transformers import AutoModelForSequenceClassification

df = pd.read_csv('/content/drive/MyDrive/ParsCN.csv')
df['Annotator'] = df['Annotator'].replace('GP-4o', 'GPT-4o')
df = df.dropna(subset=['Counter_Narrative'])

df_sampled = df.groupby('Annotator').apply(lambda x: x.sample(n=min(n_samples, len(x)), random_state=42)).reset_index(drop=True)

tox_tokenizer = AutoTokenizer.from_pretrained('textdetox/glot500-toxicity-classifier')
tox_model = AutoModelForSequenceClassification.from_pretrained('textdetox/glot500-toxicity-classifier')

def get_toxicity_score(text):
    inputs = tox_tokenizer(text, return_tensors='pt', truncation=True, padding=True)
    with torch.no_grad():
        logits = tox_model(**inputs).logits
    return torch.nn.functional.softmax(logits, dim=-1)[0][1].item()

df_sampled['toxicity_score'] = df_sampled['Counter_Narrative'].astype(str).apply(get_toxicity_score)
df_sampled.to_csv('ParsCN_125each_with_toxicity.csv', index=False)
df_sampled.groupby('Annotator')['toxicity_score'].mean().to_csv('Toxicity_by_annotator_125.csv')